# Physics informed neural network (PINN) for ODE
## Underdamped spring mass system.

#### <p style="text-align: right;"> &#9989; **put your name here** </p>

This notebook file of demonstrating PINN is designed based on a Matlab example (https://www.mathworks.com/discovery/physics-informed-neural-networks.html). **The majority of the code is generated by ChatGPT 5.0.**

---


### Part 1. Examine the data.

Here, we have the following measured data for positions at respective times. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# data points
t_data = np.array([2.51413055, 2.69057256, 0.70275129, 1.36194889, 1.18205768, 1.13500698,
 1.2026347, 0.13345253, 0.02948306, 2.3967284, 2.84302119, 0.99167425])

X_data = np.array([-0.06681997, 0.30174692, 0.36124562, -0.47074238, -0.62837142, -0.34655542,
 -0.4468897, 1.15918284,  0.94512986,  0.03274587,  0.49890506, -0.42037818])

plt.figure(figsize=(4,3))
plt.scatter(t_data, X_data, color='red')
plt.grid(True)

---
### Part 2. Using ANN to fit the data points.

Now, we have 12 data points. The input value is time and the output value is position. We will build and train a NN model to predict the object position according to time. Ideally, we can **fit a curve to those points using neuron network models**. This section is similar to the previous code demonstration artificial neural network. This model has input layer, output layer, and several hidden layers.

The cell below load the PyTorch libaray and set the device to use CPU.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

* Here, we set up a time duration of $[0, 10]$. We also normalize the time such that scaled input time is between 0 and 1. This can increase the training enstivity. Scaled time:

$$ s= t/t_{max}.$$

In [ ]:
# Time interval (seconds)
t_min, t_max = 0.0, 10

# Scaled time s = t / t_max
s_train = t_data / t_max

print(s_train)

* The data points are stored as numpy arrays. Let's convert numpy arrays to PyTorch tensor.

In [ ]:
# Convert to PyTorch tensors (shape: (N,1))
s_train_torch = torch.tensor(s_train, dtype=torch.float32).view(-1, 1).to(device)
X_train_torch = torch.tensor(X_data, dtype=torch.float32).view(-1, 1).to(device)

s_train_torch.shape, X_train_torch.shape

# s_train_torch is scaled input vector
# X_train_torch is the output values

* Set up a neural network using PyTorch libary. This NN by default has 3 hidden layers and 32 neurons per hidden layer.

In [ ]:
class Net(nn.Module):
    def __init__(self, n_hidden=3, n_neurons=32):
        super().__init__()
        layers = []
        layers.append(nn.Linear(1, n_neurons))
        layers.append(nn.Tanh())
        for _ in range(n_hidden - 1):
            layers.append(nn.Linear(n_neurons, n_neurons))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(n_neurons, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, s):
        return self.net(s)

* Set up a loss function of data. Note that this loss function is for the deviations between the NN predicted output and the true observed (measured) output. 

In [ ]:
mse = nn.MSELoss()

# loss function based on data
def data_loss(model, y_train, X_train):
    X_pred = model(y_train)
    return mse(X_pred, X_train_torch)

* Initialize the NN model, named model_NN.
* Initialize the optimizer.

In [ ]:
model_NN = Net().to(device)
optimizer = optim.Adam(model_NN.parameters(), lr=1e-3)

print(model_NN)

* Train the model with 5000 epochs. This section is the same as in the previous class.

In [ ]:
epochs = 5000

hist_data = []

for epoch in range(1, epochs + 1):
    model_NN.train()
    optimizer.zero_grad()

    Ld = data_loss(model_NN, s_train_torch, X_train_torch)
    Ld.backward()
    optimizer.step()

    hist_data.append(Ld.item())

    if epoch % 1000 == 0 or epoch == 1:
        print(f"Epoch {epoch:5d}/{epochs}, Data: {Ld.item():.4e}")

* Test the model with test data. **Note that we did not implement train-test split here.** All available data points have been used in training. Thus, we create a set of linearly spaced grid points as the time values, and use the model to predict the corresponding positions. Then, the input and output will be used to plott the predicted (fitted) curve.

In [ ]:
model_NN.eval()

# Dense time grid for the true solution
t_test = np.linspace(t_min, t_max, 101)

with torch.no_grad():
    s_plot = torch.tensor(t_test / t_max, dtype=torch.float32).view(-1, 1).to(device)
    X_pred = model_NN(s_plot).cpu().numpy().flatten()

X_pred_NN = X_pred

In [ ]:
# Plot true solution and noisy data
plt.figure(figsize=(6,4))
plt.plot(t_test, X_pred_NN, 'g-', label="NN fitted curve", linewidth=2)
plt.scatter(t_data, X_data, color="red", label="data")
plt.xlabel("time")
plt.ylabel("displacement")
plt.grid(True)
plt.legend()
plt.title("time-position data and NN fitted curve")
plt.tight_layout()
plt.show()

---
### Part 3. Underlying physics.



<img src="https://www.shimrestackor.com/Physics/Spring_Mass_Damper/Figs/2-cart.png" width="400"><a href="https://www.shimrestackor.com/Physics/Spring_Mass_Damper/spring-mass-damper.htm"><p style="text-align: right;">
Image from https://www.shimrestackor.com/Physics/Spring_Mass_Damper/spring-mass-damper.htm
</p></a>

These data points are from a measurement of **damped-spring-mass system**. According to Newton's law of force balance, the governing equation is

$$m \ddot{X}(t) + c\dot{X}(t) + k X(t) = 0,$$

where $m$ is the mass, $c$ is the damping coefficient, and $k$ is the spring constant. The first through third terms are acceleration, friction (dragging), elastic strain forces. **RECALL YOUR ENGINEERING MATH**, two coefficients are deduced from those coefficients:

$$\omega_0 = \sqrt{\frac{k}{m}}~~~\text{(undamped natural frequency)},$$

$$\zeta = \frac{c}{2\sqrt{km}}~~~\text{(damping ratio)}.$$

When $0< \zeta < 1$, it's an underdamped system, the analytical solution is 

$$X(t)=e^{-\zeta \omega_0 t} \big( A \cos(\omega_d t) + B \sin(\omega_d t ) \big),$$

where

$$\omega_d = \omega_0 \sqrt{1-\zeta^2} ~~~\text{(damped natural frequency)}.$$

The two amplitudes ($A$ and $B$) are obtained from initial conditions: 

$$A = X(0)~~~\text{and}~~~B = -\dot{X}(0).$$

#### Part 3.1. Analytical solution

* Here, let's plot the analytical solution using the cell below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# 1. Physical parameters
# -----------------------------
m = 1     # mass
c = 0.4   # damping coefficient
k = 4     # spring constant

omega0 = np.sqrt(k/m)        # natural frequency
zeta = c/(2 * np.sqrt(k*m))  # damping ratio (0 < zeta < 1 => underdamped)

omegad = omega0 * np.sqrt(1.0 - zeta**2)  # damped natural frequency 

print(f"omega0 = {omega0:.3f}, c = {c:.3f}")


# initial conditions
x0 = 1.0    # initial displacement
v0 = 0.0    # initial velocity

# time grid
t_end = 10.0
dt = 0.001          # time step (small for stability)
N = int(t_end / dt)

t = np.linspace(0.0, t_end, N+1)

# -----------------------------
# 2. Analytical solution (for comparison)
# -----------------------------
omega_d = omega0 * np.sqrt(1.0 - zeta**2)   # damped natural frequency

# general underdamped solution:
# x(t) = e^{-zeta*omega0*t} [ A cos(omega_d t) + B sin(omega_d t) ]
# use initial conditions x(0) = x0, v(0) = v0 to solve for A,B:
A = x0
B = (v0 + zeta*omega0*x0) / omega_d

x_exact = np.exp(-zeta*omega0*t) * (A * np.cos(omega_d * t) + B * np.sin(omega_d * t))

# -----------------------------
# 3. Plot the analytical solution
# -----------------------------
plt.figure(figsize=(8,4))
plt.plot(t, x_exact, 'k--', label='Analytical', linewidth=2)
plt.scatter(t_data, X_data, color='red', label='data')
plt.xlabel('time t')
plt.ylabel('displacement x(t)')
plt.title('Underdamped spring–mass system: analytical solution')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

&#9989; **Do This -** Compare to the analytical solution, does the NN fitted curve make sense to you?

---
#### Part 3.2. Numerical solution

* We can also use **finite difference method (FDM)** to numerical solve the damped-spring-mass equation (or simulate the damped oscillation).
* The stencil is given as the following. This is an ODE, in which the variable is time. The index $j$ indicates the $j$-th time step.

$$a_j = -\frac{c}{m} v_j - \frac{k}{m} X_j,$$

$$v_{j+1} = v_j + \Delta t \cdot a_j,$$

$$X_{j+1} = X_j + \Delta t \cdot v_j.$$

Hopefully, you can recall this Verlet time scheme which was used in the molecular dynamics homework assignment.

In [ ]:
# allocate arrays
x = np.zeros(N+1)
v = np.zeros(N+1)

# set initial conditions
x[0] = x0
v[0] = v0


# -----------------------------
# 4. Finite difference time stepping (forward Euler)
# -----------------------------
for n in range(N):
    # acceleration from ODE: m x'' = -c x' - k x
    a = ??

    # update velocity and displacement
    v[n+1] = ??
    x[n+1] = ??

x_num = x

In [ ]:
# -----------------------------
# 5. Plot the numerical and analytical solutions
# -----------------------------
plt.figure(figsize=(8,4))
plt.plot(t, x_exact, 'k--', label='Analytical', linewidth=2)
plt.plot(t, x_num, 'b', label='Numerical', linewidth=1)
plt.scatter(t_data, X_data, color='red', label='data')
plt.xlabel('time t')
plt.ylabel('displacement x(t)')
plt.title('Underdamped spring–mass system: finite difference vs analytical')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


&#9989; **Do This -** Do you think the finite difference mehthod did an excellent job?

&#9989; **Do This -** The available data points span only over a small section of the total time duration. Do you NN can extrapolate (predict) the displacement in the region where there are no data points being used in training?  

---
### Part 4. Physics informed neural network (PINN). 

The idea of PINN is that when the physics of the data is known, i.e., there is a physics model (governed by equations). The governing equations can be used as part of the model training. Specifically, the NN predicted values will be substituted to the giverning equations. The difference is part of the loss function. Thus, as you can expect, **the weights in NN are also adjusted to make the predictions satisfy the governing equations**. To do so, we include a physics loss term to the total loss funtion.

$$J_{total} = J_{data} + J_{phys}.$$

The solution of an ODE is dependent on its initial condition. Therefore, **if the initial condition is known, the model prediction should also match the initial condition.** In this case, we will include a loss function accounting for the initial condition. Such that the total loss is 

$$J_{total} = J_{data} + J_{phys} + J_{ic}.$$

The training is to find the values of the weight that minimize the total loss function. The rest of the tasks follow the NN process: forward propagation to prediction and backward correction of wieghts.

#### Part 4.1. Build and train PINN model
* For differential equations, we will need to calculate derivatives. This is done by using PyTorch built-in function `autograd`.

In [ ]:
# calculate derivative (gradient): dX/dt
def grad(outputs, inputs):
    return torch.autograd.grad(
        outputs,
        inputs,
        grad_outputs=torch.ones_like(outputs),
        create_graph=True)[0]

* **Calculate physics loss**. In the code cell below, we randomly select time points ($s$), substitute to the trained model to predict a displacement $\hat{X}$. Then this model predicted value is substituted to the ODE to calculate the residual (derivation from satisfying the ODE).

$$\text{model}(s) = \hat{X}.$$

$$R_{es}(t) = m \frac{d^2 \hat{X}}{d t^2} + c \frac{d \hat{X}}{d t} + k \hat{X} ~~~\Longrightarrow~~~R_{es}(s)= m \frac{d^2 \hat{X}}{d s^2} + t_{max} c \frac{d \hat{X}}{d s} + t_{max}^2 k \hat{X},$$  

since $t = s\cdot t_{max}$.

* If residual is zero, the equation is satisfied with the input variable. Note that the input variables are scaled time ($s$).

In [ ]:
def physics_loss(model, n_collocation=2000, t_max = 10, m = 1, c = 0.4, k = 4):
    # sample collocation points t in [0, t_max]
    s_f = torch.rand(n_collocation, 1, device=device)
    s_f.requires_grad_(True)

    # gradients
    X_f = model(s_f)
    dX_ds = grad(X_f, s_f)
    d2X_ds2 = grad(dX_ds, s_f)

    # ODE of damped spring mass system
    ode_res = m * d2X_ds2 + t_max * c * dX_ds + t_max**2 * k * X_f
    return torch.mean(ode_res**2)

* Initial condition loss. If you have the initial condition ($A$ and $B$), the model will need to match the initial conditions. Below is a loss function for initial conditions.

In [ ]:
def ic_loss(model, t_max = 10, x0=0.0, v0=0.0):
    s0 = torch.zeros(1, 1, device=device, requires_grad=True)  # s=0 => t=0
    X0 = model(s0)
    dX_ds0 = grad(X0, s0)
    dX_dt0 = dX_ds0 / t_max  # because t = s * t_max

    loss_x0 = (X0 - x0)**2
    loss_v0 = (dX_dt0 - v0)**2
    return (loss_x0 + loss_v0).mean()

* Total loss function. Now, we include governing equation as an additional reference to calculate derivation between NN predicted output and true value (both measured data and governing equation). Note that in the code below, we introduce a coefficient (lambda_phys) as a weighing factor between data loss and physics loss.

In [ ]:
def total_loss(model, m=1.0, c=0.4, k=4.0, lambda_phys=1.0e-4, 
               t_max=10, lambda_ic=1.0e-4, x0 = 1.0, v0 = 0.0 ):
    
    # data loss, physics loss, and IC loss 
    L_data = data_loss(model, s_train_torch, X_train_torch)
    L_phys = physics_loss(model, n_collocation=2000, t_max=10, m=m, c=c, k=k)
    L_ic   = ic_loss(model, t_max=10, x0=x0, v0=v0)  # choose ICs you want

    # total loss
    L = L_data + lambda_phys * L_phys + lambda_ic * L_ic
    return L, L_data.detach(), L_phys.detach(), L_ic.detach()

* **Finally, let's put everything together.** We initialize a PINN model that has the same structure of the previous NN model. 

In [ ]:
s_train_torch = torch.tensor(s_train, dtype=torch.float32).view(-1, 1).to(device)
X_train_torch = torch.tensor(X_data, dtype=torch.float32).view(-1, 1).to(device)

model_PINN = Net().to(device)
optimizer = optim.Adam(model_PINN.parameters(), lr=1e-3)

* Train the PINN model.

In [ ]:
# -----------------------------
# Physical parameters
# -----------------------------
m = 1     # mass
c = 0.4   # damping coefficient
k = 4     # spring constant
lambda_phys = 0.5e-4    # weigh-in of physics
lambda_ic=1.0e-4        # weigh-in of IC

# initial condition
x0 = 1.0
v0 = 0.0

# -----------------------------
# Train the PINN model
# -----------------------------
epochs = 5000

n_epo = []
hist_total, hist_data, hist_phys, hist_ic = [], [], [], []

for epoch in range(1, epochs + 1):
    model_PINN.train()
    optimizer.zero_grad()
    
    L, Ld, Lp, Li = total_loss(model_PINN, m, c, k, 
                           lambda_phys, t_max, 
                           lambda_ic, x0, v0)

    L.backward()
    optimizer.step()

    hist_total.append(L.item())
    hist_data.append(Ld.item())
    hist_phys.append(Lp.item())
    hist_ic.append(Li.item())
    n_epo.append(epoch)

    if epoch % 1000 == 0 or epoch == 1:
        print(f"Epoch {epoch:5d}/{epochs} |", 
              f"total: {L.item():.4e},", 
              f"Data: {Ld.item():.4e},",
              f"phys: {Lp.item():.4e},", 
              f"ic: {Li.item():.4e}")

&#9989; **Do This -** The PINN model takes much longer in model training. Use your own words to explain why.

---
* Observe the loss from different contributions versus epoch. 

In [ ]:
# plot loss function versus epoch
plt.figure(figsize=(4,3))
plt.plot(n_epo, hist_total, label="total")
plt.plot(n_epo, hist_data, label="data")
plt.plot(n_epo, lambda_phys*np.array(hist_phys), label="phys")
plt.plot(n_epo, lambda_ic*np.array(hist_ic), label="ic")
plt.xlabel("epoch")
plt.ylabel("losses")
plt.legend()
plt.grid(True)
plt.title("loss from different sources")

---
* Evaluate the trained PINN model. Again, we use a set of linearly spaced grid to plot the fitted curve.

In [ ]:
model_PINN.eval()

# Dense time grid for the true solution
t_test = np.linspace(t_min, t_max, 101)

with torch.no_grad():
    s_plot = torch.tensor(t_test / t_max, dtype=torch.float32).view(-1, 1).to(device)
    X_pred = model_PINN(s_plot).cpu().numpy().flatten()

X_pred_PINN = X_pred    

In [ ]:
# Plot true solution and noisy data
plt.figure(figsize=(6,4))
plt.plot(t_test, X_pred_PINN, color="magenta", label="PINN fitted curve", linewidth=2)
plt.scatter(t_data, X_data, color="red", label="data")
plt.xlabel("time")
plt.ylabel("displacement")
plt.grid(True)
plt.legend()
plt.title("PINN fitting")
plt.tight_layout()
plt.show()

---
#### Part 4.2. Comparison between PINN and NN results.

The training data are available in the time range between 0 and 3. The model training beyond available data points are purely based on physics (ODE). We will compare the predictability between the regular NN model and PINN model

* Let's plot the two curves and compare the results from NN and PINN model. 

In [ ]:
# Plot true solution and noisy data
plt.figure(figsize=(6,4))
plt.plot(t_test, X_pred_PINN,color="magenta", label="PINN fitted curve", linewidth=2)
plt.plot(t_test, X_pred_NN, 'g--', label="NN fitted curve", linewidth=1)
plt.plot(t, x_exact, 'k--', label='Analytical', linewidth=2)
plt.plot(t, x_num, 'b', label='Numerical', linewidth=1)

plt.scatter(t_data, X_data, color="red", label="data")
plt.xlabel("time")
plt.ylabel("displacement")
plt.grid(True)
plt.legend()
plt.title("Comparison of all results")
plt.tight_layout()
plt.show()

&#9989; **Do This -** The accuracy of PINN model could be bad in the tail part compared to the numerical solution. Use your own words to explain why.

# Please upload your completed notebook file via this [LINK](https://www.dropbox.com/request/hu676x256w36e11qtrru).